# 06 — Avaliação operacional

Fecha a fase 7 com os cenários D1–D3 do estudo de melhorias. Uma única divisão temporal (notebook 05) produz um único
número por modelo. Aqui:

1. **Várias origens no tempo (D1):** em cada data de corte, o modelo treina com todo o passado e é testado nos
   6 meses seguintes. É o que acontece em produção, repetido 7 vezes.
2. **Métrica de decisão (D2):** quantas faltas o serviço alcança alertando 5%, 10% ou 20% dos agendamentos.
3. **Subgrupos (D3):** para quem o modelo funciona e para quem não funciona.

**Configurações comparadas:**

| Configuração | Variáveis | ID | Ajuste de nível |
|---|---|---|---|
| Linha de base | v1 (25, notebook 02) | exato | não |
| Fase 7 | v7 (38: + histórico recente, suavizado, partida a frio) | aproximado | não |
| Fase 7 + ajuste | v7 | aproximado | sim (taxa dos últimos 6 meses) |

Hiperparâmetros da regressão logística e do LightGBM: escolhidos em cada origem, na validação interna temporal
(últimos 25% do passado), com 15 tentativas do Optuna. `periodo` fica de fora, porque o futuro nunca tem um valor
visto no treino. Requisitos: RF-060 a RF-063 (`REQUISITOS.md`).

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from noshow.features import CATEGORICAL, CATEGORICAL_V3, CONTINUOUS, CONTINUOUS_V3
from noshow.operational import decision_table, run_origins, subgroup_auc
from noshow.registry import MODELOS, load_bases

warnings.filterwarnings("ignore")
assert os.environ.get("PYTHONHASHSEED") == "0", "Use o kernel do projeto (PYTHONHASHSEED=0); ver README."
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAIDA = RAIZ / "resultados" / "avaliacao_operacional"
RECALCULAR = False  # True refaz as 7 origens (~30–40 min)

ORIGENS = ["2017-07-01", "2018-01-01", "2018-07-01", "2019-01-01", "2019-07-01", "2021-01-01", "2021-07-01"]
CONFIGS = {
    "Linha de base": {"base": "exato", "categoricas": CATEGORICAL, "continuas": CONTINUOUS},
    "Fase 7": {"base": "aproximado", "categoricas": CATEGORICAL_V3, "continuas": CONTINUOUS_V3},
    "Fase 7 + ajuste": {"base": "aproximado", "categoricas": CATEGORICAL_V3, "continuas": CONTINUOUS_V3,
                        "ajuste_nivel": {"meses": 6, "minimo": 500}},
}
bases = load_bases()
if RECALCULAR or not (SAIDA / "predicoes.parquet").exists():
    M, P = run_origins(bases, CONFIGS, ORIGENS)
    SAIDA.mkdir(parents=True, exist_ok=True)
    M.to_csv(SAIDA / "metricas.csv", index=False)
    P.to_parquet(SAIDA / "predicoes.parquet", index=False)
else:
    M, P = pd.read_csv(SAIDA / "metricas.csv"), pd.read_parquet(SAIDA / "predicoes.parquet")

BLUE, ORANGE, GRAY, INK, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#52514e", "#1baf7a"
COR_CFG = {"Linha de base": GRAY, "Fase 7": BLUE, "Fase 7 + ajuste": ORANGE}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
M.groupby("origem")[["n_treino", "n_teste", "taxa_teste"]].first().style.format(
    {"n_treino": "{:,}", "n_teste": "{:,}", "taxa_teste": "{:.1%}"})

## 1. Desempenho em cada origem (D1)

In [ ]:
fig, axes = plt.subplots(1, len(MODELOS), figsize=(16, 3.8), sharey=True)
for ax, m in zip(axes, MODELOS):
    for cfg, cor in COR_CFG.items():
        s = M[(M.modelo == m) & (M.config == cfg)]
        ax.plot(s.origem.astype(str), s.roc_auc, "o-", color=cor, lw=2, ms=6, mec="white", mew=1.5, label=cfg)
    ax.axhline(0.5, color=GRAY, ls="--", lw=1)
    ax.set_title(m); ax.tick_params(axis="x", rotation=45, labelsize=8)
axes[0].set_ylabel("ROC-AUC"); axes[-1].legend(frameon=False, fontsize=8)
fig.suptitle("ROC-AUC nos 6 meses seguintes a cada origem", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

resumo = M.groupby(["modelo", "config"]).agg(roc_media=("roc_auc", "mean"), roc_dp=("roc_auc", "std"),
                                              pr_media=("pr_auc", "mean"), brier=("brier", "mean"))
resumo.loc[[(m, c) for m in MODELOS for c in CONFIGS]].style.format("{:.4f}") \
    .highlight_max(subset=["roc_media", "pr_media"], color="#d6e6f8")

**Comparações pareadas por origem** (Wilcoxon, 7 pares; o menor *p* bilateral possível é 0,016).

In [ ]:
linhas = []
for m in MODELOS:
    piv = M[M.modelo == m].pivot_table(index="origem", columns="config", values="roc_auc")
    for a, b in [("Fase 7", "Linha de base"), ("Fase 7 + ajuste", "Fase 7")]:
        d = piv[a] - piv[b]
        linhas.append({"modelo": m, "comparação": f"{a} × {b}", "diferença_média": d.mean(),
                       "origens_a_maior": int((d > 0).sum()), "p": stats.wilcoxon(piv[a], piv[b]).pvalue})
pd.DataFrame(linhas).set_index(["modelo", "comparação"]).style.format(
    {"diferença_média": "{:+.4f}", "p": "{:.3f}"}).map(lambda v: "font-weight: bold" if v < 0.05 else "", subset=["p"])

## 2. Métrica de decisão (D2)

Em produção, o serviço alerta os pacientes de maior risco **dentro do período que está prevendo**. Por isso, a tabela
é calculada **em cada origem** (alertando os 5%, 10% ou 20% de maior risco daqueles 6 meses) e depois é feita a média
ponderada pelo tamanho do teste. Juntar as predições de todas as origens e ordená-las juntas premiaria quem acerta o
nível de risco *entre* períodos, uma comparação que o serviço nunca faz.

In [ ]:
melhor = M.groupby(["config", "modelo"]).roc_auc.mean().groupby("config").idxmax().map(lambda t: t[1])
tabelas = []
for cfg, m in melhor.items():
    for origem, s in P[(P.config == cfg) & (P.modelo == m)].groupby("origem"):
        tabelas.append(decision_table(s.y.to_numpy(), s.p.to_numpy()).assign(config=cfg, modelo=m, n=len(s)))
t = pd.concat(tabelas)
cols = ["alertas_por_1000", "recall", "precisao", "ganho", "faltas_alcancadas_por_1000", "faltas_ao_acaso_por_1000"]
dec = t.groupby(["config", "modelo", "volume"]).apply(lambda g: pd.Series(np.average(g[cols], axis=0, weights=g.n), cols))
dec.style.format({"alertas_por_1000": "{:.0f}", "recall": "{:.1%}", "precisao": "{:.1%}", "ganho": "{:.2f}×",
                  "faltas_alcancadas_por_1000": "{:.1f}", "faltas_ao_acaso_por_1000": "{:.1f}"})

## 3. Subgrupos (D3)

Desempenho da melhor configuração por grupo de pacientes, calculado em cada origem e combinado pela média ponderada
pelo tamanho do grupo (pelo mesmo motivo da seção 2). Em cada origem, grupos com menos de 100 agendamentos ou sem as
duas classes ficam de fora.

In [ ]:
cfg_final = M.groupby("config").roc_auc.mean().idxmax()
m_final = melhor[cfg_final]
s = P[(P.config == cfg_final) & (P.modelo == m_final)].reset_index(drop=True)
info = bases["aproximado"].iloc[s.linha.to_numpy()].reset_index(drop=True)
hist = pd.cut(info.n_agendamentos_anteriores, [-1, 0, 4, 9, 29, 10_000],
              labels=["0 (novo)", "1–4", "5–9", "10–29", "30+"]).astype("string").fillna("sem ID")
print(f"Configuração: {cfg_final} · modelo: {m_final} · {len(s):,} agendamentos previstos")
grupos = {"faixa_etaria": info.faixa_etaria, "specialty_grp": info.specialty_grp, "histórico": hist,
          "id_qualidade": info.id_qualidade}

def por_origem(g):
    partes = []
    for _, idx in s.groupby("origem").groups.items():
        idx = np.asarray(idx)
        partes.append(subgroup_auc(s.y.to_numpy()[idx], s.p.to_numpy()[idx], g.iloc[idx], minimo=100))
    t = pd.concat(partes).reset_index()
    return t.groupby("grupo").apply(lambda d: pd.Series({
        "n": d.n.sum(), "taxa": np.average(d.taxa, weights=d.n),
        "roc_auc": np.average(d.roc_auc.dropna(), weights=d.n[d.roc_auc.notna()]) if d.roc_auc.notna().any() else np.nan,
        "origens": int(d.roc_auc.notna().sum())}))

pd.concat({nome: por_origem(g) for nome, g in grupos.items()}).style.format(
    {"n": "{:,.0f}", "taxa": "{:.1%}", "roc_auc": "{:.3f}", "origens": "{:.0f}"}, na_rep="—") \
    .background_gradient(subset=["roc_auc"], cmap="RdBu", vmin=0.4, vmax=0.7)

## 4. Síntese

### Resultado

| Comparação (ROC-AUC média em 7 origens) | Linha de base | Fase 7 | Diferença | Origens melhores |
|---|---|---|---|---|
| **LightGBM** | 0,534 | **0,549** | **+0,016 (*p* = 0,016)** | **7 de 7** |
| Regressão logística | 0,542 | **0,558** | +0,016 (*p* = 0,078) | 5 de 7 |
| RB (AIC) | 0,510 | 0,524 | +0,014 (n.s.) | 4 de 7 |
| RB (BIC) | 0,515 | 0,510 | −0,005 (n.s.) | 1 de 7 |

- **As variáveis da fase 7 melhoram de forma consistente.** Com o LightGBM, o ganho aparece nas 7 origens, o
  resultado mais robusto de todos os experimentos.
- **O ajuste de nível não muda nada aqui:** métricas idênticas com e sem ajuste, porque a correção é monotônica e a
  avaliação é feita dentro de cada período.
- **Prevendo os 6 meses seguintes, a discriminação real é fraca:** ROC-AUC de 0,51 a 0,56. As AUCs maiores da divisão
  temporal única (~0,57) incluíam o efeito do nível de risco *entre* períodos.

### O que isso vale para o serviço (melhor modelo: regressão logística, fase 7)

| Alertando os de maior risco | Faltas alcançadas por 1.000 agendamentos | Sorteando | Ganho |
|---|---|---|---|
| 5% | 9,2 | 5,4 | 1,75× |
| 10% | 15,6 | 10,8 | 1,52× |
| 20% | 27,4 | 21,5 | 1,33× |

Comparada à linha de base, a fase 7 alcança cerca de **2 faltas a mais por 1.000 agendamentos** com 10% de alertas
(15,6 contra 13,7), com o mesmo volume de contatos.

### Para quem o modelo não funciona

- **Pacientes novos** (AUC 0,497) e **sem ID** (0,485) ficam no nível do acaso. Sem histórico, não há sinal
  suficiente, e as variáveis de partida a frio não resolveram.
- **Idosos (60+)** (0,515) e **adolescentes** (0,534) ficam perto do acaso; a primeira infância (0–5, 0,581) e a
  fonoaudiologia (0,583) são os grupos mais previsíveis.
- Isso deve entrar no TCC como **limitação de equidade**: o modelo ajuda mais quem já tem histórico no serviço.

### Conclusão da fase 7

| Cenário do estudo | Resultado |
|---|---|
| C1/C2 — ajuste do nível de risco (v6) | **Não ajudou**: instável com a taxa mensal volátil; irrelevante para alertas por volume |
| B1–B3 — histórico suavizado, recência, partida a frio (v7) | **Ajudou pouco, mas de forma consistente**: +0,016 em 7/7 origens; melhor agrupada 0,690 |
| B4 — rótulo limpo (v8) | **Não ajudou**: ruído pequeno demais para mudar o modelo |
| D1–D3 — avaliação operacional (06) | Feita; mostra ganho real de 1,5× sobre o acaso com 10% de alertas e os grupos onde o modelo falha |

O teto continua dado pela informação disponível. O próximo salto depende dos cenários A1/A2 do estudo: a data de
marcação e um ID real de paciente.